# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/Users/katexxi/Desktop/AIEC1/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
load_dotenv()

gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.16
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/var/folders/fw/kwy0s5tj0l39fzpsqntd04nm0000gn/T/ipykernel_13570/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [8]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [9]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████| 20/20 [00:41<00:00,  2.09s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]/Users/katexxi/Desktop/AIEC1/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [01:22<00:00,  1.37s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 320.46it/s]

KnowledgeGraph(nodes: 20, relationships: 52)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [19]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")
    # entities = node.properties.get("entities", [])
    # print(f"entities: {len(entities)}\n{entities}")
    # summary = node.properties.get("summary", "")
    # print(f"summary: \n{summary}")
    # themes = node.properties.get("themes", [])
    # print(f"themes: \n{themes}\n")

Node types:
- NodeType.CHUNK: 20
Relationships: 52
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [11]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 52)


#### ❓ Question #1

##### ✅ Answer

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

Graph transforms add value by converting unstructured document chunks into a structured knowledge graph enriched with entities, themes, summaries, embeddings, and relationships while preserving the original text. 

The two relationships capture both local connections within a chunk and broader connections across chunks through shared entities, themes, or semantic similarity, which is important for multi-hop question as it requires connecting information across multiple concepts or documents. 

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [20]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [21]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

SingleHopSpecificQuerySynthesizer: focuses on a single fact or piece of evidence from one chunk and does not require combining information across multiple contexts.

MultiHopSpecificQuerySynthesizer: connect multiple facts, which requires combining multiple related facts from different chunks by following explicit relationships between them.

MultiHopAbstractQuerySynthesizer: connect multiple ideas from themes from each chuck, which requires synthesizing multiple related contexts or themes across chunks to answer a broader conceptual question rather than simply asking for facts.

Of course it's multiHopAbstractQuerySynthesizer as tests its abilities of retrieving multiple contexts plus reasoning at a higher conceptual level.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [22]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [00:13<00:00,  2.29s/it]


,user_input,reference,synthesizer_name
0,Wha article in the Journal of the American Ani...,The guidelines are published in the Journal of...,single_hop_specific_query_synthesizer
1,Why is a life stage assessment important for c...,A life stage assessment is important because t...,single_hop_specific_query_synthesizer
2,"How do ""Neutering and behavioral effects"" and ...",The context shows that handling and social exp...,multi_hop_abstract_query_synthesizer
3,"According to the ""Veterinary healthcare guidel...",The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_abstract_query_synthesizer
4,How ACVIM fit with the cat care guidelines and...,The context says the 2018 ACVIM consensus stat...,multi_hop_specific_query_synthesizer
5,According to the AAHA/AAFP Feline Life Stage G...,The AAHA/AAFP Feline Life Stage Guidelines rec...,multi_hop_specific_query_synthesizer


In [23]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

The one-call path is simpler and requires fewer lines of code because Ragas handles the entire testset generation workflow automatically. It is useful when a standard evaluation dataset is sufficient and no special customization is needed. Besides giving us the ability to inspect & debug each stage of the pipeline, unrolled path can also support query type distribution customization. 

I would choose the one-call path for quick dataset generation and the unrolled path when I need more control over the evaluation dataset or want to inspect intermediate results.




## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [27]:
# Activity #1 workspace
#
# Start with every generated example. Replace this with your reviewed subset.
approved_testset_df = testset_df.copy()
review_status = "review_required"

# Examples:
approved_testset_df = testset_df.drop(index=[2, 4]).reset_index(drop=True)
approved_testset_df.loc[0, "user_input"] = "Which JAAHA article published the guidelines and what did it say about cat life stages?"
review_status = "student_reviewed"

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

,user_input,reference_contexts,reference,synthesizer_name
0,Which JAAHA article published the guidelines a...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...,The guidelines are published in the Journal of...,single_hop_specific_query_synthesizer
1,Why is a life stage assessment important for c...,[Introduction\nThe feline patient ’s life stag...,A life stage assessment is important because t...,single_hop_specific_query_synthesizer
2,"According to the ""Veterinary healthcare guidel...",[<1-hop>\n\nIntroduction\nThe feline patient ’...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_abstract_query_synthesizer
3,According to the AAHA/AAFP Feline Life Stage G...,[<1-hop>\n\ndetection of changes and identi ﬁc...,The AAHA/AAFP Feline Life Stage Guidelines rec...,multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes

- Example reviewed:
- Decision:
- Reason:
- Any safety or grounding issue found:

Example 1: "Wha article in the Journal of the American Animal Hospital Association is the guidelines in, and what did they say about cat life stages?"; Edit the question to "Which JAAHA article published the guidelines and what did it say about cat life stages?"; It's a valid useful question with answer from reference context directly but we need to fix the gramma and make it sounded more natural; No

Exmaple 2: "Why is a life stage assessment important for cats during each examination visit, especially in the United States?"; Keep it; This is a clean single-hop question. Direct Answer matches with the reference context as well; No

Exmaple 3: "How do "Neutering and behavioral effects" and "Socialization and handling effects" together explain changes in cats’ behavior toward people and other cats?; Remove it; The question itself is only partially answerable, because the the reference contexts is just a citation list with some titles relevant to what's asked. It didn't provide enough retrieved evidence. Besides it sounds not like what a cat ownder will ask with mentioning the themes directly; No


Exmaple 4: "According to the "Veterinary healthcare guidelines" and "Veterinary practice guidelines" in the 2021 AAHA/AAFP Feline Life Stage Guidelines, what is the life stage framework used for feline healthcare, and what practical components are included in a wellness visit to support an individualized, lifelong approach as a cat matures?"; Keep it (even tho the asked query is a bit artificial); The two part of this question are all totally answerable from provided context and the answer is generated directly from that; No


Exmaple 5: "How ACVIM fit with the cat care guidelines and zoonoses stuff here?"; Remove it; The question itself is only partially answerable because there is no direct connection in between ACVIM and zoonoses in the reference context. Therefore even tho the answer is grounded with the context but it didn't fully answer the question; No

Exmaple 6: "According to the AAHA/AAFP Feline Life Stage Guidelines, how should veterinarians combine life-stage examination focus and nutrition/weight management when evaluating kittens, young adult cats, and senior cats in relation to detecting trends and preventing obesity or elimination problems?"; Keep it; The question is fully answerable by combining all the reference context and the answer is grounded to the evidences; No 


---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [28]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-1141cb2f
Examples uploaded: 4


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

synthesizer_name tells us which query synthesizer was used to generate the example. Keeping this information makes it easier to trace unexpected or low-quality questions back to their source and compare the performance of different synthesizers.

synthetic_reference helps with filtering, auditing, and debugging the dataset. It also makes it easier to compare evaluation results across different datasets and understand whether differences are caused by the model or by the types of examples included in the evaluation set.

review_status keeps track of whether a generated example has been reviewed and approved by a human. This makes it easier to filter high-quality examples, revisit problematic ones, and monitor the overall quality of the dataset over time.


## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [29]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [30]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [31]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [32]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The corpus says a feline wellness visit should consider:

- Behavior and social interaction
- Environmental and enrichment needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes important related topics such as feline-friendly handling practices, overcoming barriers to examination visits, environmental enrichment, understanding feline behavior, practice team training, and client education.

The corpus does not provide more detailed component-by-component guidance beyond that.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [33]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [34]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

One example is when the model gives an answer that is fully supported by the retrieved context but is still incorrect because the context itself is incomplete or missing important information. In this case, groundedness would be high while correctness would be low. The disagreement here suggests inspecting the retrieval trace, including the retrieved contexts and chunk selection process, to determine whether relevant evidence was missing.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [35]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-61d3986a' at:
https://smith.langchain.com/o/fcdd1cb0-04e8-4c18-a6df-cc7026724ff0/datasets/e6624d6a-748a-4f46-85ce-e07ddfe55dbd/compare?selectedSessions=ec25be76-64ca-4036-8508-d0baa62e93bb




4it [00:15,  3.89s/it]

Baseline experiment: cat-health-rag-baseline-k3-61d3986a


### Baseline Inspection Notes

- Lowest-scoring example:

Query: According to the AAHA/AAFP Feline Life Stage Guidelines, how should veterinarians combine life-stage examination focus and nutrition/weight management when evaluating kittens, young adult cats, and senior cats in relation to detecting trends and preventing obesity or elimination problems?

Output: The retrieved context does not provide enough information to answer that question fully. What it does say is: 
The guidelines recommend a life stage assessment at each examination visit. 
They advise a minimum of annual examinations for all cats, with more frequent visits as needed based on individual needs.
For cats with weight gain, it is a good time to evaluate nutritional needs, obesity risks, and prevention strategies.
The guidelines are meant to support individualized care recommendations.
Some senior cats aged 10 years and older may still be in excellent condition and could be treated as mature adults at the veterinarian’s discretion. However, the retrieved text does not include enough detail about how veterinarians should specifically combine life-stage exam focus with nutrition/weight management for kittens, young adult cats, and senior cats in relation to detecting trends or preventing obesity or elimination problems.

- Metric that failed:

answer_correctness: 0.4; retrieval_relevance: 0.67

- Was the synthetic reference valid?

Yes

- Was the retrieved context relevant and sufficient?

Retrieved context: ["Page: 2\nSource: cat_health_guidelines.pdf\nshort period of time, each examination visit should include a life\nstage assessment. The 2021 AAHA/AAFP Feline Life Stage Guidelines\nprovide a comprehensive age-associated framework for promoting\nhealth and longevity throughout a cat ’s lifetime. The guidelines were\ndeveloped by a T ask Force of experts in feline clinical medicine.\nTheir recommendations are a practical resource to guide individu-\nalized risk assessment, preventive healthcare strategies, and treat-","Page: 6\nSource: cat_health_guidelines.pdf\nFor example, some senior cats aged 10 years and older may remain in\nexcellent physical condition and would be best treated as a mature\nadult at the veterinarian ’s discretion. The guidelines are intended to\nbe a starting point from which individualized care recommenda-\ntions can be developed.\nDiscussion Items for All Life Stages\nThe Task Force recommends a minimum of annual examinations for\nall cats, with increasing frequency as appropriate for their individual\nneeds.","Page: 13\nSource: cat_health_guidelines.pdf\nwith weight gain,\n89 this is an excellent time to evaluate the nutri-\ntional needs, obesity risks, and prevention strategies for the indi-\nvidual patient. Recommendations can be found in the AAFP ’s Feline\nFeeding Programs Consensus Statement.\n19\nYoung Adult Cats\nEnergy requirements of cats are in ﬂuenced by a variety of factors\nincluding age (i.e., life stage), BCS, MCS, neuter status, health status,\nand activity level. Using indirect calorimetry, young adult active cats"]

It's relevant but not sufficient at all. It did answered a bit information related to life-stage assessments, individualized care, nutrition and obesity prevention but didn't provide any on kittens, young adult cats, and senior cats in relation to detecting trends or preventing obesity or elimination problems.


- Did the answer add unsupported information?

No, all the information can point to supported evidence in retrieved context.

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [36]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- **History and life stage assessment**
- **Environmental and social needs**
- **Elimination**
- **Nutrition and weight management**
- **Oral health**
- **Parasite control**
- **Vaccination**
- **Zoonoses and human safety**
- **Diagnostics**

It also notes additional important topics for the visit, including:

- **Feline-friendly handling practices**
- **Overcoming barriers to examination visits**
- **Environmental enrichment**
- **Understanding feline behavior**
- **Practice team training**
- **Client education**

If you want, I can also summarize how these topics connect to different cat life stages using only the provided guidelines.

Retrieved context count: 6


In [37]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-b5ff3e95' at:
https://smith.langchain.com/o/fcdd1cb0-04e8-4c18-a6df-cc7026724ff0/datasets/e6624d6a-748a-4f46-85ce-e07ddfe55dbd/compare?selectedSessions=f7205dc0-c7aa-43b1-8858-e865b29b3361




4it [00:16,  4.22s/it]

Candidate experiment: cat-health-rag-candidate-k6-b5ff3e95


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

Because if we change multiple variables at once, we won't be able to tell what caused the result change. The larger retrieval set may be providing additional useful evidence that helps answer the question correctly, even though some of the extra chunks are less relevant. 

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [39]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75
RETRIEVAL_K = 3

def make_rag_target(chunk_size: int = CHUNK_SIZE):
    # build a new vector store
    rag_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=CHUNK_OVERLAP,
    )
    rag_documents = rag_splitter.split_documents(source_documents)
    rag_embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL_NAME,
        api_key=gateway_api_key,
        base_url=GATEWAY_BASE_URL,
        check_embedding_ctx_length=False,
    )
    vector_store = QdrantVectorStore.from_documents(
        documents=rag_documents,
        embedding=rag_embeddings,
        location=":memory:",
        collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
    )
    print(f"Source PDF pages: {len(source_documents)}")
    print(f"RAG chunks: {len(rag_documents)}")
    
    retriever = vector_store.as_retriever(
        search_kwargs={"k": RETRIEVAL_K}
    )
    
    rag_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", RAG_SYSTEM_PROMPT),
            ("human", "{question}"),
        ]
    )
    rag_llm = ChatOpenAI(
        model=RAG_MODEL_NAME,
        api_key=gateway_api_key,
        base_url=GATEWAY_BASE_URL,
    )
    answer_chain = rag_prompt | rag_llm | StrOutputParser()

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": RETRIEVAL_K,
            "chunk_size": chunk_size,
            "chunk_overlap": CHUNK_OVERLAP,
        }

    target.__name__ = f"cat_health_rag_{chunk_size}_{CHUNK_OVERLAP}_{RETRIEVAL_K}"
    return target

In [44]:
additional_baseline_spot_check = make_rag_target(chunk_size=1500)(
    {"question": "What components should be considered during a feline wellness visit?"}
)
print(additional_baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in additional_baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

Source PDF pages: 20
RAG chunks: 83
The corpus says a feline wellness visit should consider:

- A detailed medical history, including previous medical/surgical history and any past or current medications or supplements
- Assessment of the cat’s current diet, including amount, feeding frequency, and how the cat is fed
- Nutritional recommendations to continue or change the current diet
- Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS)
- The cat’s temperament, demeanor, and handling preferences
- Observation of how the cat is reacting to the environment
- Discussion of expected normal behaviors for the cat’s life stage
- Review of subtle signs of anxiety, illness, and pain
- Discussion of preventive healthcare and follow-up modifications at later exams
- For adult and senior cats, questions about appetite changes, polyuria/polydipsia, vomiting, hairballs, diarrhea, increased nocturnal activity, vocalization, and changes in habits or a

In [ ]:
new_chunk_size = 750
additional_target = make_rag_target(new_chunk_size)
additional_results = evaluate(
    additional_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        f"Candidate cat health RAG with the same index and "
        f"chunk size updated from {CHUNK_SIZE} to {new_chunk_size}."
    ),
    metadata={
        "chunk_size": new_chunk_size,
        "chunk_overlap": CHUNK_OVERLAP,
        "retrieval_k": RETRIEVAL_K,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"additional experiment: {additional_results.experiment_name}")

Source PDF pages: 20
RAG chunks: 164
View the evaluation results for experiment: 'cat-health-rag-candidate-k6-4fb0f08d' at:
https://smith.langchain.com/o/fcdd1cb0-04e8-4c18-a6df-cc7026724ff0/datasets/e6624d6a-748a-4f46-85ce-e07ddfe55dbd/compare?selectedSessions=97ca2588-fe9f-43c1-b66a-6d9c9af7a7e4




4it [00:17,  4.41s/it]

additional experiment: cat-health-rag-candidate-k6-4fb0f08d


### 📝 Activity #2 Notes

- Baseline result: answer_correctness: 0.51; answer_groundedness: 0.95; retrieval_relevance: 0.76

- Candidate result: answer_correctness: 0.56; answer_groundedness: 0.95; retrieval_relevance: 0.88

- Two traces inspected: 

First: Why is a life stage assessment important for cats during each examination visit, especially in the United States? 

answer_correctness 
0.75 -> 0.70
answer_groundedness 
1.00 -> 0.88
retrieval_relevance 
0.95 -> 0.98 

This is the only query we're seeing both answer_correctness & answer_groundedness decreases after increasing the retrieval value. I noticed that the first three received context are the same. However, only one of the additional three retrieved later is relevant to the question with mentioning "Kittens will have different health risks depending on their lifestyle
and history" where the unrelevant ones have brought unnecessary noises to the model. 

Second: According to the "Veterinary healthcare guidelines" and "Veterinary practice guidelines" in the 2021 AAHA/AAFP Feline Life Stage Guidelines, what is the life stage framework used for feline healthcare, and what practical components are included in a wellness visit to support an individualized, lifelong approach as a cat matures? 

answer_correctness 
0.40 -> 0.60
answer_groundedness 
1.00 -> 1.00
retrieval_relevance 
0.80 -> 0.82 

This the another example we have with really low score on answer_correctness when context window is too small. Two of the additional contexts retrieved are citiations which is not helpful but the other one "... an evolving, individualized, lifelong healthcare strategy for each feline patient at every life stage." is matching what's asked here.

- Decision: k=6 increases both the average score of answer_correctness & retrieval_relevance. I think it does improve the application overall.

- Cost or latency tradeoff: We're not seeing any major differences in latency (2.44 vs 2.31) in this comparison but it should increase with increase the k value because extra time will be needed for processing more contexts from the model. 
The total token increased from 2312 to 3696 which totally makes sense because now the context window is so much bigger. 

- Variable changed: chunk_size (500 -> 750 -> 1000 -> 1250)

- Prediction: answer_correctness should definitely increase with increasing the chunk size. There shouldn't be too much changes on answer_groundedness & retrieval_relevance

- Third experiment result:

answer_correctness 
0.40 -> 0.84 -> 0.83 -> 0.80
answer_groundedness 
1.00 -> 0.88 -> 0.90 -> 0.95
retrieval_relevance 
0.80 -> 0.93 -> 0.92 -> 0.98

Correctness score got increase for all the given examples, especially for failure ones (they're all passing now). It got increased from 0.40 to 0.95 for the second example we mentioned above, because the previous chunk size was too small with truncating so much important information. However, after increasing the chunk size to 1250, correctness score dropped for all the examples as well, and that's because more context also introduce extra irrelevant information to the model.

retrieval_relevance also got increased a lot and that's also because previous chunk size was too small, making it harder for retrieval to capture the full context needed to answer a question.

There are slightly changes on answer_groundedness but overall I think it's pretty stable, which make senses as well as it measures whether the generated answer is supported by the retrieved context, not whether retrieval quality improves. As long as model can get enough evidence from retrieved context to generate answer, increasing chunk size won't change it that much. 



## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.